# Phase 4 — Posterior Predictive Checks

This notebook checks whether the fitted latent mood and user-taste models can generate realistic data. We compare posterior-predictive song features, mood frequencies, listen probabilities, and binned calibration against the real Phase 0 data. The implementation stays within the project scope: Bayesian mixtures, Dirichlet-Categorical structure, Bernoulli likelihoods, SVI, and posterior predictive simulation.


## 0  Imports and global configuration

We use the same import order, seed, and plotting style as the previous project notebooks. The notebook is self-contained: it loads the Phase 0 cleaned CSV files, reconstructs the Phase 1 tensors, refits the Phase 1 mood model, refits the Phase 3 user-taste model, and then performs posterior predictive checks.


In [ ]:
import numpy as np
import torch
import pyro
import pyro.distributions as dist
import matplotlib.pyplot as plt
import seaborn as sns

import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from pyro.infer import SVI, TraceEnum_ELBO, Trace_ELBO, Predictive, infer_discrete, config_enumerate
from pyro.infer.autoguide import AutoDiagonalNormal
from pyro.infer.autoguide.initialization import init_to_value
from pyro.optim import Adam
from pyro.ops.indexing import Vindex

np.random.seed(67)
torch.manual_seed(67)
pyro.set_rng_seed(67)

plt.style.use('ggplot')
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)
sns.set_context('talk')

print(f"pyro {pyro.__version__}, torch {torch.__version__}, numpy {np.__version__}")


## 1  Load Phase 0 cleaned data

Phase 0 is the source of truth for the cleaned corpus. We load `songs_clean.csv` and `listens_clean.csv` from the project `data/` directory. The song table provides the observed audio features; the listen table provides the observed implicit-feedback events for calibration and listen-pattern checks.


In [ ]:
REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / 'data'
SONGS_CSV = DATA_DIR / 'songs_clean.csv'
LISTENS_CSV = DATA_DIR / 'listens_clean.csv'

assert SONGS_CSV.exists(), f"{SONGS_CSV} not found — run phase0_data.ipynb first."
assert LISTENS_CSV.exists(), f"{LISTENS_CSV} not found — run phase0_data.ipynb first."

df_songs = pd.read_csv(SONGS_CSV)
df_listens = pd.read_csv(LISTENS_CSV)

print(f"Songs loaded:   {len(df_songs):,}")
print(f"Listens loaded: {len(df_listens):,}")
print("Song columns:", df_songs.columns.tolist())
print("Listen columns:", df_listens.columns.tolist())


## 2  Reconstruct Phase 1 model inputs

We rebuild the exact mixed-likelihood representation used by Phase 1. The continuous features are z-scored, while `key`, `time_signature`, and `mode` are treated according to their actual data type. This keeps Phase 4 aligned with the already fitted Phase 1 and Phase 3 models.


In [ ]:
# --- Continuous features: z-score ---
CONT_COLS = ['loudness', 'tempo', 'duration']
scaler = StandardScaler()
X_cont_np = scaler.fit_transform(df_songs[CONT_COLS].values).astype(np.float32)

# --- key: recover integer 0-11 from MinMax representation ---
KEY_NAMES = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
N_KEY = 12
X_key_np = (df_songs['key'] * 11).round().astype(int).clip(0, 11).values

# --- time_signature: map retained values to 0..N_TS-1 ---
ts_vals = sorted(df_songs['time_signature'].unique())
ts_to_idx = {v: i for i, v in enumerate(ts_vals)}
X_ts_np = df_songs['time_signature'].map(ts_to_idx).values.astype(int)
N_TS = len(ts_vals)
TS_LABELS = ['ts=0','ts=1','ts=3','ts=4','ts=5','ts=7'] if N_TS == 6 else [f'ts={v}' for v in ts_vals]

# --- mode: binary 0/1 ---
X_mode_np = df_songs['mode'].values.astype(np.float32)

# --- Torch tensors ---
X_cont = torch.tensor(X_cont_np, dtype=torch.float32).cpu()
X_key = torch.tensor(X_key_np, dtype=torch.long).cpu()
X_ts = torch.tensor(X_ts_np, dtype=torch.long).cpu()
X_mode = torch.tensor(X_mode_np, dtype=torch.float32).cpu()
N, D = X_cont.shape

print(f"X_cont : {tuple(X_cont.shape)}")
print(f"X_key  : {tuple(X_key.shape)}  range={X_key.min().item()}..{X_key.max().item()}")
print(f"X_ts   : {tuple(X_ts.shape)}   range={X_ts.min().item()}..{X_ts.max().item()}")
print(f"X_mode : {tuple(X_mode.shape)} range={X_mode.min().item()}..{X_mode.max().item()}")

assert torch.isfinite(X_cont).all()
assert X_key.min() >= 0 and X_key.max() < N_KEY
assert X_ts.min() >= 0 and X_ts.max() < N_TS
assert X_mode.min() >= 0 and X_mode.max() <= 1


## 3  Fit the Phase 1 mood model

The posterior predictive check needs posterior samples of the global mood parameters. We therefore fit the same mixed-likelihood Bayesian mixture model used in Phase 1. The latent mood assignment $z_s$ is enumerated in SVI, and the guide approximates the posterior over the continuous global parameters.


In [ ]:
K = 10  # selected in Phase 1 by the ΔELBO elbow

MOOD_NAMES = [
    'Loud mainstream',
    'Loud fast',
    'Dark minor',
    'Bright mainstream',
    'Acoustic folk/blues',
    'Loud slow',
    'Long midtempo',
    'Long atmospheric',
    'Quiet slow ballad',
    'Ambient long-form',
]

def mood_model(X_cont, X_key, X_ts, X_mode, K, mu_prior_loc=None):
    D = X_cont.shape[1]
    device = X_cont.device

    pi = pyro.sample("pi", dist.Dirichlet(5.0 * torch.ones(K, device=device)))

    with pyro.plate("moods", K):
        mu_cont = pyro.sample(
            "mu_cont",
            dist.Normal(torch.zeros(D, device=device), torch.ones(D, device=device)).to_event(1)
        )
        sigma_cont = pyro.sample(
            "sigma_cont",
            dist.LogNormal(torch.zeros(D, device=device), 0.5 * torch.ones(D, device=device)).to_event(1)
        )
        theta_key = pyro.sample("theta_key", dist.Dirichlet(torch.ones(N_KEY, device=device)))
        theta_ts = pyro.sample("theta_ts", dist.Dirichlet(torch.ones(N_TS, device=device)))
        p_mode = pyro.sample(
            "p_mode",
            dist.Beta(torch.tensor(2.0, device=device), torch.tensor(2.0, device=device))
        )

    with pyro.plate("songs", X_cont.shape[0]):
        z = pyro.sample("z", dist.Categorical(pi), infer={"enumerate": "parallel"})

        mu_z = Vindex(mu_cont)[z]
        sigma_z = Vindex(sigma_cont)[z]
        theta_key_z = Vindex(theta_key)[z]
        theta_ts_z = Vindex(theta_ts)[z]
        p_mode_z = Vindex(p_mode)[z]

        pyro.sample("obs_cont", dist.Normal(mu_z, sigma_z).to_event(1), obs=X_cont)
        pyro.sample("obs_key", dist.Categorical(theta_key_z), obs=X_key)
        pyro.sample("obs_ts", dist.Categorical(theta_ts_z), obs=X_ts)
        pyro.sample("obs_mode", dist.Bernoulli(p_mode_z), obs=X_mode)

km = KMeans(n_clusters=K, n_init=20, random_state=67).fit(X_cont_np)
kmeans_centers = torch.tensor(km.cluster_centers_, dtype=torch.float32)

print(f"K={K}")
print("K-means cluster sizes:", np.bincount(km.labels_, minlength=K))


In [ ]:
def fit_mood_svi(seed, n_steps=1500, lr=1e-2):
    pyro.set_rng_seed(seed)
    pyro.clear_param_store()

    init_vals = {
        "mu_cont": kmeans_centers,
        "pi": torch.ones(K) / K,
        "sigma_cont": torch.ones(K, D),
        "p_mode": 0.5 * torch.ones(K),
        "theta_key": torch.ones(K, N_KEY) / N_KEY,
        "theta_ts": torch.ones(K, N_TS) / N_TS,
    }

    guide = AutoDiagonalNormal(
        pyro.poutine.block(mood_model, hide=["z"]),
        init_loc_fn=init_to_value(values=init_vals),
        init_scale=0.05,
    )

    svi = SVI(
        mood_model,
        guide,
        Adam({"lr": lr}),
        TraceEnum_ELBO(max_plate_nesting=2),
    )

    losses = []
    for step in range(n_steps):
        loss = svi.step(X_cont, X_key, X_ts, X_mode, K, kmeans_centers)
        losses.append(loss)
        if step % 250 == 0:
            print(f"seed={seed} step={step:4d} loss={loss:,.0f}")
    return guide, losses

best = None
mood_runs = []
for seed in (555, 363, 126, 82):
    g, l = fit_mood_svi(seed)
    mood_runs.append((seed, g, l))
    print(f"seed={seed} final loss={l[-1]:,.0f}")
    if best is None or l[-1] < best[2][-1]:
        best = (seed, g, l)

best_seed, mood_guide, mood_losses = best
mood_median = mood_guide.median()
print(f"\nBest seed: {best_seed}, final ELBO loss: {mood_losses[-1]:,.0f}")


### 3.1  MAP mood assignments

For the user-taste likelihood and mood-frequency PPC, we need one mood label per song. We use the same MAP extraction pattern as Phase 1 and Phase 3: replay the fitted guide into the model and run `infer_discrete` at temperature zero.


In [ ]:
guide_trace = pyro.poutine.trace(mood_guide).get_trace(X_cont, X_key, X_ts, X_mode, K, kmeans_centers)
trained_model = pyro.poutine.replay(mood_model, trace=guide_trace)
map_model = infer_discrete(config_enumerate(trained_model), temperature=0, first_available_dim=-2)
map_trace = pyro.poutine.trace(map_model).get_trace(X_cont, X_key, X_ts, X_mode, K, kmeans_centers)
z_map = map_trace.nodes['z']['value'].detach().cpu().numpy()

mood_counts = np.bincount(z_map, minlength=K)
print("MAP mood counts:")
for k, cnt in enumerate(mood_counts):
    print(f"  {k:2d} {MOOD_NAMES[k]:<22s}: {cnt:>7,} ({cnt / len(z_map) * 100:5.1f}%)")


## 4  Fit the Phase 3 user-taste model

Phase 3 models each user's taste as a Dirichlet distribution over the fixed mood assignments. We refit that model here so the posterior predictive checks can compare predicted listen behaviour with the observed `listened` labels. The mood model is not retrained inside the user model; the listen likelihood conditions on the MAP mood assignment $z_s$.


In [ ]:
# Build song_id → corpus index and corpus index → mood mapping.
df_songs = df_songs.copy()
df_songs['song_idx'] = np.arange(len(df_songs))
song_id_to_idx = df_songs.set_index('song_id')['song_idx'].to_dict()
song_idx_to_mood = dict(zip(range(len(df_songs)), z_map))

# Keep listen events whose song_id is in the song corpus.
df_listens_model = df_listens[df_listens['song_id'].isin(song_id_to_idx)].copy()
df_listens_model['song_idx'] = df_listens_model['song_id'].map(song_id_to_idx)
df_listens_model['mood'] = df_listens_model['song_idx'].map(song_idx_to_mood).astype(int)

# Active-user filter, matching Phase 3.
user_counts = df_listens_model.groupby('user_id').size()
active_users = user_counts[user_counts >= 5].index
df_listens_model = df_listens_model[df_listens_model['user_id'].isin(active_users)].copy()

users = sorted(df_listens_model['user_id'].unique())
user_to_idx = {u: i for i, u in enumerate(users)}
df_listens_model['user_idx'] = df_listens_model['user_id'].map(user_to_idx)
U = len(users)

user_idx_t = torch.tensor(df_listens_model['user_idx'].values, dtype=torch.long)
mood_idx_t = torch.tensor(df_listens_model['mood'].values, dtype=torch.long)
listened_t = torch.tensor(df_listens_model['listened'].values, dtype=torch.float32)

print(f"Users modelled: {U:,}")
print(f"Listen events:  {len(df_listens_model):,}")
print(f"Positive listens: {int(df_listens_model['listened'].sum()):,}")
print(f"Negative listens: {int((df_listens_model['listened'] == 0).sum()):,}")
print("Mood counts in listens:", df_listens_model['mood'].value_counts().sort_index().to_dict())


In [ ]:
ALPHA = 0.5

def taste_model(user_idx, mood_idx, listened, U, K):
    with pyro.plate("users", U):
        theta = pyro.sample("theta", dist.Dirichlet(ALPHA * torch.ones(K)))

    with pyro.plate("listens", len(listened)):
        prob = torch.sigmoid(K * theta[user_idx, mood_idx])
        pyro.sample("obs", dist.Bernoulli(prob), obs=listened)


def taste_guide(user_idx, mood_idx, listened, U, K):
    alpha_q = pyro.param(
        "alpha_q",
        ALPHA * torch.ones(U, K),
        constraint=dist.constraints.positive,
    )
    with pyro.plate("users", U):
        pyro.sample("theta", dist.Dirichlet(alpha_q))

pyro.clear_param_store()
pyro.set_rng_seed(67)

svi_taste = SVI(taste_model, taste_guide, Adam({"lr": 1e-2}), Trace_ELBO())
N_TASTE_STEPS = 1000
losses_taste = []

for step in range(N_TASTE_STEPS):
    loss = svi_taste.step(user_idx_t, mood_idx_t, listened_t, U, K)
    losses_taste.append(loss)
    if step % 100 == 0:
        print(f"taste step={step:4d} loss={loss:,.0f}")

alpha_q = pyro.param("alpha_q").detach()
theta_post = (alpha_q / alpha_q.sum(dim=1, keepdim=True)).cpu().numpy()
print("theta_post shape:", theta_post.shape)


## 5  Posterior predictive simulation

We draw approximate posterior samples of the mood parameters from the SVI guide and then generate fresh synthetic songs by ancestral sampling. For listens, we use the variational posterior mean of each user's taste profile and simulate binary listen labels for the same observed `(user, mood)` pairs. This gives direct fake-vs-real comparisons for both song features and user behaviour.


In [ ]:
N_PPC_SAMPLES = 100
N_PPC_SONGS = min(10000, N)

# Draw posterior samples of global mood parameters from the SVI guide.
param_predictive = Predictive(
    mood_model,
    guide=mood_guide,
    num_samples=N_PPC_SAMPLES,
    return_sites=("pi", "mu_cont", "sigma_cont", "theta_key", "theta_ts", "p_mode"),
)

posterior_params = param_predictive(X_cont, X_key, X_ts, X_mode, K, kmeans_centers)

for name, value in posterior_params.items():
    print(name, tuple(value.shape))


In [ ]:
def sample_synthetic_songs(params, n_songs):
    """Generate posterior-predictive songs from one posterior parameter sample."""
    pi = params['pi']
    mu_cont = params['mu_cont']
    sigma_cont = params['sigma_cont']
    theta_key = params['theta_key']
    theta_ts = params['theta_ts']
    p_mode = params['p_mode']

    z = dist.Categorical(pi).sample((n_songs,))
    x_cont = dist.Normal(mu_cont[z], sigma_cont[z]).sample()
    x_key = dist.Categorical(theta_key[z]).sample()
    x_ts = dist.Categorical(theta_ts[z]).sample()
    x_mode = dist.Bernoulli(p_mode[z]).sample()
    return z, x_cont, x_key, x_ts, x_mode

fake_cont_all = []
fake_key_all = []
fake_ts_all = []
fake_mode_all = []
fake_mood_all = []

for s in range(N_PPC_SAMPLES):
    params_s = {name: val[s].detach() for name, val in posterior_params.items()}
    z_fake, cont_fake, key_fake, ts_fake, mode_fake = sample_synthetic_songs(params_s, N_PPC_SONGS)
    fake_mood_all.append(z_fake.cpu().numpy())
    fake_cont_all.append(cont_fake.cpu().numpy())
    fake_key_all.append(key_fake.cpu().numpy())
    fake_ts_all.append(ts_fake.cpu().numpy())
    fake_mode_all.append(mode_fake.cpu().numpy())

fake_cont_all = np.stack(fake_cont_all)   # (samples, n_songs, D)
fake_key_all = np.stack(fake_key_all)
fake_ts_all = np.stack(fake_ts_all)
fake_mode_all = np.stack(fake_mode_all)
fake_mood_all = np.stack(fake_mood_all)

print("fake_cont_all:", fake_cont_all.shape)
print("fake_mood_all:", fake_mood_all.shape)


## 6  PPC: continuous audio-feature histograms

The first check compares the real z-scored continuous feature distributions to the posterior-predictive distributions. Good overlap means the learned mixture can reproduce the marginal shapes of loudness, tempo, and duration. Clear mismatches suggest that the diagonal Gaussian mixture is too simple or that some components are underfit.


In [ ]:
real_idx = np.random.default_rng(67).choice(N, size=N_PPC_SONGS, replace=False)
real_cont = X_cont_np[real_idx]
fake_cont_flat = fake_cont_all.reshape(-1, D)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for d, (ax, col) in enumerate(zip(axes, CONT_COLS)):
    sns.histplot(real_cont[:, d], bins=60, stat='density', color='black', alpha=0.35, label='real', ax=ax)
    sns.histplot(fake_cont_flat[:, d], bins=60, stat='density', color='tab:blue', alpha=0.35, label='posterior predictive', ax=ax)
    ax.set_title(f'PPC: {col}')
    ax.set_xlabel(f'{col} (z-scored)')
    ax.legend()
plt.tight_layout()
plt.show()


## 7  PPC: categorical and binary audio features

We next compare generated and observed distributions for key, time signature, and mode. These checks target the Dirichlet-Categorical and Beta-Bernoulli parts of the model. Since these variables are discrete, bar plots are more informative than density curves.


In [ ]:
def posterior_predictive_category_summary(fake_arr, n_categories):
    props = []
    for sample in fake_arr:
        props.append(np.bincount(sample.astype(int), minlength=n_categories) / len(sample))
    props = np.stack(props)
    return props.mean(0), np.percentile(props, 5, axis=0), np.percentile(props, 95, axis=0)

key_mean, key_lo, key_hi = posterior_predictive_category_summary(fake_key_all, N_KEY)
ts_mean, ts_lo, ts_hi = posterior_predictive_category_summary(fake_ts_all, N_TS)
mode_mean, mode_lo, mode_hi = posterior_predictive_category_summary(fake_mode_all.astype(int), 2)

real_key = np.bincount(X_key_np, minlength=N_KEY) / len(X_key_np)
real_ts = np.bincount(X_ts_np, minlength=N_TS) / len(X_ts_np)
real_mode = np.bincount(X_mode_np.astype(int), minlength=2) / len(X_mode_np)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

x = np.arange(N_KEY)
axes[0].bar(x - 0.2, real_key, width=0.4, label='real', color='black', alpha=0.55)
axes[0].bar(x + 0.2, key_mean, width=0.4, label='posterior predictive', color='tab:blue', alpha=0.65)
axes[0].errorbar(x + 0.2, key_mean, yerr=[key_mean - key_lo, key_hi - key_mean], fmt='none', color='tab:blue')
axes[0].set_xticks(x)
axes[0].set_xticklabels(KEY_NAMES, rotation=45)
axes[0].set_title('PPC: key')
axes[0].legend()

x = np.arange(N_TS)
axes[1].bar(x - 0.2, real_ts, width=0.4, label='real', color='black', alpha=0.55)
axes[1].bar(x + 0.2, ts_mean, width=0.4, label='posterior predictive', color='tab:blue', alpha=0.65)
axes[1].errorbar(x + 0.2, ts_mean, yerr=[ts_mean - ts_lo, ts_hi - ts_mean], fmt='none', color='tab:blue')
axes[1].set_xticks(x)
axes[1].set_xticklabels(TS_LABELS, rotation=45)
axes[1].set_title('PPC: time signature')
axes[1].legend()

x = np.arange(2)
axes[2].bar(x - 0.2, real_mode, width=0.4, label='real', color='black', alpha=0.55)
axes[2].bar(x + 0.2, mode_mean, width=0.4, label='posterior predictive', color='tab:blue', alpha=0.65)
axes[2].errorbar(x + 0.2, mode_mean, yerr=[mode_mean - mode_lo, mode_hi - mode_mean], fmt='none', color='tab:blue')
axes[2].set_xticks(x)
axes[2].set_xticklabels(['minor', 'major'])
axes[2].set_title('PPC: mode')
axes[2].legend()

plt.tight_layout()
plt.show()


## 8  PPC: mood-frequency distribution

The mood-frequency check compares the MAP mood distribution in the real corpus with the synthetic mood assignments drawn from the posterior predictive mixture weights. This tests whether the learned $\boldsymbol{\pi}$ produces a similar balance of moods to the fitted real corpus.


In [ ]:
mood_mean, mood_lo, mood_hi = posterior_predictive_category_summary(fake_mood_all, K)
real_mood = np.bincount(z_map, minlength=K) / len(z_map)

x = np.arange(K)
plt.figure(figsize=(14, 6))
plt.bar(x - 0.2, real_mood, width=0.4, label='real MAP moods', color='black', alpha=0.55)
plt.bar(x + 0.2, mood_mean, width=0.4, label='posterior predictive', color='tab:blue', alpha=0.65)
plt.errorbar(x + 0.2, mood_mean, yerr=[mood_mean - mood_lo, mood_hi - mood_mean], fmt='none', color='tab:blue')
plt.xticks(x, [f'{k}\n{MOOD_NAMES[k]}' for k in range(K)], rotation=45, ha='right')
plt.ylabel('Proportion')
plt.title('PPC: mood-frequency distribution')
plt.legend()
plt.tight_layout()
plt.show()


## 9  PPC: listen probabilities and calibration

The user-taste model predicts a listen probability for each observed `(user, song)` pair through the user's taste weight for that song's mood. We compare predicted probabilities against observed listen labels. The calibration plot bins predictions and checks whether the empirical listen frequency matches the predicted probability in each bin.


In [ ]:
theta_t = torch.tensor(theta_post, dtype=torch.float32)
with torch.no_grad():
    pred_prob = torch.sigmoid(K * theta_t[user_idx_t, mood_idx_t]).cpu().numpy()

obs_listen = listened_t.cpu().numpy()

plt.figure(figsize=(10, 6))
sns.histplot(pred_prob[obs_listen == 1], bins=40, stat='density', alpha=0.45, label='observed listened=1')
sns.histplot(pred_prob[obs_listen == 0], bins=40, stat='density', alpha=0.45, label='observed listened=0')
plt.xlabel('Predicted listen probability')
plt.title('Predicted listen probabilities by observed label')
plt.legend()
plt.tight_layout()
plt.show()

# Calibration by probability bins.
calib_df = pd.DataFrame({'pred_prob': pred_prob, 'observed': obs_listen})
calib_df['bin'] = pd.qcut(calib_df['pred_prob'], q=10, duplicates='drop')
calib = calib_df.groupby('bin', observed=True).agg(
    pred_mean=('pred_prob', 'mean'),
    empirical_rate=('observed', 'mean'),
    n=('observed', 'size'),
).reset_index()

plt.figure(figsize=(8, 8))
plt.plot([0, 1], [0, 1], 'k--', label='perfect calibration')
plt.scatter(calib['pred_mean'], calib['empirical_rate'], s=np.sqrt(calib['n']) * 2, label='bins')
for _, row in calib.iterrows():
    plt.text(row['pred_mean'], row['empirical_rate'], str(int(row['n'])), fontsize=8)
plt.xlabel('Mean predicted probability')
plt.ylabel('Empirical listen frequency')
plt.title('Calibration plot for listen likelihood')
plt.legend()
plt.tight_layout()
plt.show()

print(calib.to_string(index=False))


## 10  PPC: listen-count distribution per user

Finally, we simulate listen labels for the observed user-song candidate pairs and compare the number of positive listens per user. This tests whether the user-taste model can reproduce heterogeneity in user activity under the same observed candidate set.


In [ ]:
N_LISTEN_PPC = 100
rng = np.random.default_rng(67)

real_counts = df_listens_model.groupby('user_idx')['listened'].sum().reindex(range(U), fill_value=0).values
fake_counts_all = []

for _ in range(N_LISTEN_PPC):
    fake_l = rng.binomial(1, pred_prob).astype(int)
    fake_df = pd.DataFrame({'user_idx': df_listens_model['user_idx'].values, 'fake_l': fake_l})
    fake_counts = fake_df.groupby('user_idx')['fake_l'].sum().reindex(range(U), fill_value=0).values
    fake_counts_all.append(fake_counts)

fake_counts_all = np.stack(fake_counts_all)
fake_counts_flat = fake_counts_all.reshape(-1)

plt.figure(figsize=(12, 6))
sns.histplot(real_counts, bins=50, stat='density', color='black', alpha=0.45, label='real')
sns.histplot(fake_counts_flat, bins=50, stat='density', color='tab:blue', alpha=0.35, label='posterior predictive')
plt.xlabel('Positive listens per user')
plt.title('PPC: listen-count distribution per user')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Real listen counts: median={np.median(real_counts):.1f}, mean={np.mean(real_counts):.1f}")
print(f"Fake listen counts: median={np.median(fake_counts_flat):.1f}, mean={np.mean(fake_counts_flat):.1f}")


## 11  Phase 4 gate summary

The gate is qualitative but we still compute simple numeric summaries. We expect the posterior-predictive distributions to overlap the real feature distributions, the mood frequencies to be close, and the listen calibration curve to be reasonably near the diagonal. Large deviations should be discussed as model limitations rather than hidden.


In [ ]:
# Simple quantitative checks for the summary.
feature_mean_real = real_cont.mean(axis=0)
feature_mean_fake = fake_cont_flat.mean(axis=0)
feature_std_real = real_cont.std(axis=0)
feature_std_fake = fake_cont_flat.std(axis=0)

feature_mean_abs_diff = np.abs(feature_mean_real - feature_mean_fake)
feature_std_abs_diff = np.abs(feature_std_real - feature_std_fake)

mood_l1 = np.abs(real_mood - mood_mean).sum()
calib_mae = np.mean(np.abs(calib['pred_mean'] - calib['empirical_rate']))

print("=" * 70)
print("PHASE 4 GATE SUMMARY — posterior predictive checks")
print("=" * 70)
print(f"Posterior predictive samples: {N_PPC_SAMPLES}")
print(f"Synthetic songs per sample:   {N_PPC_SONGS:,}")
print(f"Listen PPC replicates:        {N_LISTEN_PPC}")
print()
for d, col in enumerate(CONT_COLS):
    print(f"{col:<10s} mean diff={feature_mean_abs_diff[d]:.3f}  std diff={feature_std_abs_diff[d]:.3f}")
print()
print(f"Mood-frequency L1 distance:   {mood_l1:.3f}")
print(f"Calibration MAE:              {calib_mae:.3f}")
print()
print("Interpretation:")
print("- Good PPC: real and fake histograms overlap, mood frequencies are close, calibration points are near diagonal.")
print("- Weak PPC: systematic shifts in feature histograms, missing rare categories, or calibration far from diagonal.")


## 12  Save Phase 4 outputs

We save the main numeric PPC summaries to `data/phase4_processed/` so the report can reference them without rerunning the full notebook. The plots remain in the notebook output.


In [ ]:
PHASE4_OUT_DIR = DATA_DIR / 'phase4_processed'
PHASE4_OUT_DIR.mkdir(parents=True, exist_ok=True)

summary_rows = []
for d, col in enumerate(CONT_COLS):
    summary_rows.append({
        'check': 'continuous_feature',
        'name': col,
        'real_mean': float(feature_mean_real[d]),
        'fake_mean': float(feature_mean_fake[d]),
        'mean_abs_diff': float(feature_mean_abs_diff[d]),
        'real_std': float(feature_std_real[d]),
        'fake_std': float(feature_std_fake[d]),
        'std_abs_diff': float(feature_std_abs_diff[d]),
    })

summary_rows.append({'check': 'mood_frequency', 'name': 'L1', 'value': float(mood_l1)})
summary_rows.append({'check': 'listen_calibration', 'name': 'MAE', 'value': float(calib_mae)})

df_ppc_summary = pd.DataFrame(summary_rows)
df_ppc_summary.to_csv(PHASE4_OUT_DIR / 'phase4_ppc_summary.csv', index=False)
calib.to_csv(PHASE4_OUT_DIR / 'phase4_calibration_bins.csv', index=False)

np.save(PHASE4_OUT_DIR / 'real_mood_proportions.npy', real_mood)
np.save(PHASE4_OUT_DIR / 'fake_mood_proportions_mean.npy', mood_mean)
np.save(PHASE4_OUT_DIR / 'fake_mood_proportions_lo.npy', mood_lo)
np.save(PHASE4_OUT_DIR / 'fake_mood_proportions_hi.npy', mood_hi)

print("Saved Phase 4 outputs to:")
print(PHASE4_OUT_DIR)
for p in sorted(PHASE4_OUT_DIR.iterdir()):
    print('-', p.name)
